In [268]:
import random
import pandas as pd
import numpy as np
import copy
from collections import Counter, defaultdict

In [269]:
POPULASI = 1000
VIOLATION_COST = 100
ITERATION = 100
MUTATION_PROB = 0.7
TOURNAMENT_SIZE = 10

In [270]:
guru_df = pd.read_csv('../dataset/guru.csv')
kelas_df = pd.read_csv('../dataset/kelas.csv')
mapel_df = pd.read_csv('../dataset/mapel.csv')
relasi_guru_mapel_df = pd.read_csv('../dataset/relasi_guru_mapel.csv')
slot_df = pd.read_csv('../dataset/slot.csv')
wali_kelas_df = pd.read_csv('../dataset/wali_kelas.csv')

# DICT

In [271]:
# =========================================================
# Hari
hariId = {
"Senin": 1,
"Selasa": 2,
"Rabu": 3,
"Kamis": 4,
"Jumat": 5,
}
# hariId

# ========================================================
# mencari slot tiap per hari
# key = hariId, value = jumlah slot integer
# contoh output: {1: 8, 2: 8, 3: 8, 4: 7, 5: 5}

slotPerHari = {} # dictionary kosongan
for _, row in slot_df.iterrows():
    hari = row["hari"] # ambil kolom hari saja

    if hari not in slotPerHari:
        slotPerHari[hari] = 1
    else:
        slotPerHari[hari] += 1

slotPerHari = {hariId[k]: v for k,v in slotPerHari.items()}
# print(slotPerHari)


# ========================================================
# guru dan nama
# key = guru_id, value = nama guru
guruPengajar = dict(
    zip(guru_df['guru_id'], guru_df['nama_guru'])
)
# guruPengajar

# ========================================================
# mapping nama kelas dan tingkatan
# ada 27 kelas, contoh output: {1: [{'tingkatan': 7, 'nama_kelas': '7A'}], 2: [{'tingkatan': 7, 'nama_kelas': '7B'}]}
kelasDanTingkatan = defaultdict(list)
for _, row in kelas_df.iterrows():
    kelasDanTingkatan[row['kelas_id']].append({
        'tingkatan': row['tingkatan'],
        'nama_kelas': row['nama_kelas']
    })
kelasDanTingkatan = dict(kelasDanTingkatan)
# print(kelasDanTingkatan)

# ========================================================
# mapping nama mapel dan id
# key = mapel_id, value = nama mapel
namaMapelDanId = dict(
    zip(mapel_df['mapel_id'], mapel_df['nama_mapel'])
)
# namaMapelDanId

# =========================================================
# mencari jam per minggu tiap mape
# key = mapel_id, value = jam per minggu integerl
jamPerMingguMapel = dict(
    zip(mapel_df['mapel_id'], mapel_df['jam_per_minggu'])
)
# print(jamPerMingguMapel)

# =========================================================
# mapel id dan hari MGMP
# contoh output: {1: 1, 2: 2, 3: 2, 4: 4, 5: 4, 6: 3, 7: 1, 8: 1, 9: 3, 10: 4, 11: 5, 12: 3, 13: 2}
mgmpMapel = dict(
    zip(mapel_df['mapel_id'], mapel_df['MGMP'])
)
mgmpMapel = {mapel_id: hariId[hari] for mapel_id, hari in mgmpMapel.items()}
# print(mgmpMapel)
# =========================================================
# batas siang dan batas MGMP
# key = hariId value = slot ke berapa dalam hari tersebut
batasSiang = {1: 5, 2: 5, 3: 4, 4: 5, 5: 4}
batasMGMP = {1: 2, 2: 2, 3: 2, 4: 2, 5: 1}
# =========================================================
# durasi guru mengajar
# key = guru value = list of dict {mapel_id, tingkatan, durasi}
durasiGuruMengajar = defaultdict(list)

for _, row in relasi_guru_mapel_df.iterrows():
    durasiGuruMengajar[row["guru_id"]].append({
        "mapel_id": row["mapel_id"],
        "tingkatan": row["tingkatan"],
        "durasi": row["durasi"]
    })
durasiGuruMengajar = dict(durasiGuruMengajar)
# durasiGuruMengajar


# =========================================================
# Wali kelas guru
# key = guru_id, value = kelas_id
waliKelas = dict(
    zip(wali_kelas_df['guru_id'], wali_kelas_df['kelas_id'])
)
waliKelas

{21: 1,
 17: 24,
 19: 3,
 18: 4,
 36: 5,
 2: 6,
 31: 7,
 37: 8,
 38: 9,
 22: 10,
 34: 11,
 23: 12,
 13: 13,
 14: 14,
 32: 15,
 24: 16,
 16: 17,
 35: 18,
 8: 19,
 6: 20,
 15: 21,
 25: 22,
 20: 23,
 3: 25,
 5: 26,
 27: 27}

# INDIVIDU

In [272]:
# memecah jam_per_minggu menjadi blok yang bisa didistribusikan
def blokDistribusi(jam):
    if jam == 2:
        return [2]
    if jam == 3:
        return [3]
    if jam == 4:
        return [2,2]
    if jam == 5:
        return [2,3]
    
    return [jam]

In [273]:
# mengambil guru berdsakan mapel dan tingaktan
def ambilGuruValid(mapel_id, tingkatan):
    listGuru = []

    for guru_id, relasiList in durasiGuruMengajar.items():
        for relasi in relasiList:
            if relasi["mapel_id"] == mapel_id and relasi["tingkatan"] == tingkatan:
                listGuru.append(guru_id)
    return listGuru

In [274]:
# membuat jadwal kosongan dulu
def jadwalKosongan():
    jadwal = {}
    for hari in slotPerHari.keys():
        jadwal[hari] = []

    return jadwal

In [275]:
def slotTersedia(jadwal_kelas, hari, durasi):
    slotTerpakai = 0

    for event in jadwal_kelas[hari]:
        slotTerpakai += event["durasi"]
    
    if slotTerpakai + durasi <= slotPerHari[hari]:
        return True
    
    return False

In [276]:
def putEvent(jadwal_kelas, event):

    listHari = list(slotPerHari.keys())

    random.shuffle(listHari)

    for hari in listHari:

        if slotTersedia(jadwal_kelas, hari, event["durasi"]):
            jadwal_kelas[hari].append(event)
            
            return True
        
    return False

In [277]:
def perluasBlok(mapel_id, guru_id, durasi):
    slot = []

    for _ in range(durasi):
        slot.append({
            "mapel": mapel_id,
            "guru": guru_id
        })
    return slot

In [278]:
def generatePerKelas(tingkatan):

    pilihan = []

    for mapel_id, jam in jamPerMingguMapel.items():

        blok = blokDistribusi(jam)

        guruValid = ambilGuruValid(mapel_id, tingkatan)

        if not guruValid:
            continue

        guru = random.choice(guruValid)

        for durasi in blok:

            slot = perluasBlok(mapel_id, guru, durasi)

            pilihan.extend(slot)

    random.shuffle(pilihan)

    return pilihan

In [279]:
def individuConstruct(kelas_id):

    tingkatan = kelasDanTingkatan[kelas_id][0]["tingkatan"]

    slots = generatePerKelas(tingkatan)

    jadwal = {}

    index = 0

    for hari in slotPerHari:

        jumlahSlot = slotPerHari[hari]

        jadwal[hari] = slots[index:index+jumlahSlot]

        index += jumlahSlot

    return jadwal

In [280]:
def individuTrigger():
    individu = {}

    for kelas_id in kelasDanTingkatan.keys():

        jadwalKelas = individuConstruct(kelas_id)

        individu[kelas_id] = jadwalKelas 

    return individu

In [281]:
def populasiContruct(POPULASI):
    populasi = []

    for _ in range(POPULASI):

        individu = individuTrigger()

        populasi.append(individu)

    return populasi

In [282]:
populasiOptimasi = populasiContruct(POPULASI)

In [283]:
populasiOptimasi[0][1]

{1: [{'mapel': 7, 'guru': 21},
  {'mapel': 7, 'guru': 21},
  {'mapel': 4, 'guru': 19},
  {'mapel': 1, 'guru': 39},
  {'mapel': 5, 'guru': 48},
  {'mapel': 4, 'guru': 19},
  {'mapel': 11, 'guru': 37},
  {'mapel': 6, 'guru': 2}],
 2: [{'mapel': 10, 'guru': 15},
  {'mapel': 11, 'guru': 37},
  {'mapel': 5, 'guru': 48},
  {'mapel': 3, 'guru': 29},
  {'mapel': 2, 'guru': 39},
  {'mapel': 12, 'guru': 55},
  {'mapel': 5, 'guru': 48},
  {'mapel': 3, 'guru': 29}],
 3: [{'mapel': 12, 'guru': 55},
  {'mapel': 12, 'guru': 55},
  {'mapel': 9, 'guru': 42},
  {'mapel': 8, 'guru': 42},
  {'mapel': 3, 'guru': 29},
  {'mapel': 10, 'guru': 15},
  {'mapel': 1, 'guru': 39},
  {'mapel': 13, 'guru': 53}],
 4: [{'mapel': 4, 'guru': 19},
  {'mapel': 3, 'guru': 29},
  {'mapel': 4, 'guru': 19},
  {'mapel': 5, 'guru': 48},
  {'mapel': 7, 'guru': 21},
  {'mapel': 3, 'guru': 29},
  {'mapel': 6, 'guru': 2}],
 5: [{'mapel': 13, 'guru': 53},
  {'mapel': 8, 'guru': 42},
  {'mapel': 2, 'guru': 39},
  {'mapel': 6, 'guru':

# EVAL

In [284]:
def guruBentrok(individu):
    pelanggaran = 0

    for hari in slotPerHari:
        totalSlot = slotPerHari[hari]

        for slot in range(totalSlot):
            guruMengajar = []

            for kelas in individu:
                if slot < len(individu[kelas][hari]):
                    guru = individu[kelas][hari][slot]["guru"]
                    guruMengajar.append(guru)

                if len(guruMengajar) != len(set(guruMengajar)):
                    pelanggaran += 1
    return pelanggaran

In [285]:
def putBlokMapel(slotHari):
    blok = []

    mapelSekarang = slotHari[0]["mapel"]

    count = 1

    for i in range(1, len(slotHari)):
        if slotHari[i]["mapel"] == mapelSekarang:
            count += 1
        else:
            blok.append((mapelSekarang, count))

            mapelSekarang = slotHari[i]["mapel"]
            count = 1
    
    blok.append((mapelSekarang, count))

    return blok

def distribusiMapel(individu):
    pelanggaran = 0

    for kelas in individu:
        distribusi = {}

        for hari in individu[kelas]:
            blok = putBlokMapel(individu[kelas][hari])

            for mapel, durasi in blok:

                if mapel not in distribusi:
                    distribusi[mapel] = []

                    distribusi[mapel].append(durasi)
        for mapel in distribusi:
            jam = jamPerMingguMapel[mapel]

            if jam == 2:
                if distribusi[mapel] != [2]:
                    pelanggaran += 1
            elif jam == 3:
                if distribusi[mapel] != [3]:
                    pelanggaran += 1
            elif jam == 4:
                if sorted(distribusi[mapel]) != [2,2]:
                    pelanggaran += 1
            elif jam == 5:
                if sorted(distribusi[mapel]) != [2,3]:
                    pelanggaran += 1
    return pelanggaran

In [286]:
def mapelSiang(individu):
    pelanggaran = 0

    for kelas in individu:

        for hari in individu[kelas]:
            batas = batasSiang[hari]

            for slot in range(len(individu[kelas][hari])):
                mapel = individu[kelas][hari][slot]["mapel"]

                # if mapel == 8 and slot >= batas:
                if mapel == 8 and slot > batas:

                    pelanggaran += 1
    return pelanggaran

In [287]:
def durasiGuru(individu):
    pelanggaran = 0

    loadGuru = {}

    for kelas in individu:
        for hari in individu[kelas]:

            for slot in individu[kelas][hari]:

                guru = slot["guru"]

                if guru not in loadGuru:
                    loadGuru[guru] = 0

                loadGuru[guru] += 1
    for guru in loadGuru:
        if loadGuru[guru] > 40:
            pelanggaran += loadGuru[guru] - 40
    return pelanggaran

In [288]:
def waktuMGMP(individu):
    pelanggaran = 0

    for kelas in individu:

        for hari in individu[kelas]:
            for slot in range(len(individu[kelas][hari])):
                mapel = individu[kelas][hari][slot]["mapel"]

                if mapel in mgmpMapel:
                    hariMGMP = mgmpMapel[mapel]

                    if hari == hariMGMP:
                        if slot > batasMGMP[hari]:
                            pelanggaran += 1

    return pelanggaran

In [289]:
def cekWaliKelas(individu):
    pelanggaran = 0

    for kelas_id, jadwalKelas in individu.items():

        for hari in jadwalKelas:

            for slot in jadwalKelas[hari]:

                guru = slot['guru']

                if guru in waliKelas:

                    kelasWali = waliKelas[guru]

                    if kelas_id != kelasWali:
                        pelanggaran += 1
    return pelanggaran

In [290]:
def evaluasiIndividu(individu):

    pelanggaran = 0

    pelanggaran += guruBentrok(individu)

    pelanggaran += distribusiMapel(individu)

    pelanggaran += mapelSiang(individu)

    pelanggaran += durasiGuru(individu)

    pelanggaran += waktuMGMP(individu)

    pelanggaran += cekWaliKelas(individu)

    return pelanggaran * VIOLATION_COST

# GA

In [291]:
def crossover(parent1, parent2):
    child1 = copy.deepcopy(parent1)
    child2 = copy.deepcopy(parent2)

    hari = list(parent1.keys())

    # pilih hari yang akan ditukar
    jumlah = random.randint(1, len(hari)//2)

    hariTerpilih = random.sample(hari, jumlah)

    for h in hariTerpilih:
        child1[h], child2[h] = parent2[h], parent1[h]
        
    return child1, child2

In [292]:
def mutasi(individu, MUTATION_PROB):

    individuBaru = copy.deepcopy(individu)

    if random.random() > MUTATION_PROB:
        return individuBaru

    hari = random.choice(list(individuBaru.keys()))

    slot = individuBaru[hari]

    # jika slot adalah dict
    if isinstance(slot, dict):
        keys = list(slot.keys())

        if len(keys) < 2:
            return individuBaru

        i, j = random.sample(keys, 2)

        slot[i], slot[j] = slot[j], slot[i]

    # jika slot adalah list
    else:
        if len(slot) < 2:
            return individuBaru

        i, j = random.sample(range(len(slot)), 2)

        slot[i], slot[j] = slot[j], slot[i]

    return individuBaru

In [293]:
def turnamen(populasi, TOURNAMENT_SIZE):
    kandidat = random.sample(populasi, TOURNAMENT_SIZE)

    terbaik = min(kandidat, key=lambda x: evaluasiIndividu(x))

    return terbaik

In [294]:
def geneticAlgorithm(populasiAwal, GENERASI, POPULASI, TOURNAMENT_SIZE, MUTATION_PROB):

    populasi = copy.deepcopy(populasiAwal)

    for gen in range(GENERASI):

        populasiBaru = []

        # elitism (simpan individu terbaik)
        terbaik = min(populasi, key=lambda x: evaluasiIndividu(x))
        populasiBaru.append(copy.deepcopy(terbaik))

        while len(populasiBaru) < POPULASI:

            p1 = turnamen(populasi, TOURNAMENT_SIZE)
            p2 = turnamen(populasi, TOURNAMENT_SIZE)

            c1, c2 = crossover(p1, p2)

            c1 = mutasi(c1, MUTATION_PROB)
            c2 = mutasi(c2, MUTATION_PROB)

            populasiBaru.append(c1)

            if len(populasiBaru) < POPULASI:
                populasiBaru.append(c2)

        populasi = populasiBaru

        # evaluasi populasi
        skor = [evaluasiIndividu(ind) for ind in populasi]

        best = min(skor)
        avg = sum(skor) / len(skor)

        # if gen % 10 == 0:
        print(f"Generasi {gen} | Best: {best} | Avg: {avg:.2f}")

    # hasil akhir
    terbaik = min(populasi, key=lambda x: evaluasiIndividu(x))
    skorTerbaik = evaluasiIndividu(terbaik)

    print("\n=== HASIL AKHIR ===")
    print("Skor terbaik:", skorTerbaik)

    return terbaik

In [295]:
hasil = geneticAlgorithm(populasiOptimasi, ITERATION, POPULASI, TOURNAMENT_SIZE, MUTATION_PROB)

Generasi 0 | Best: 155500 | Avg: 164754.50
Generasi 1 | Best: 153000 | Avg: 162678.30
Generasi 2 | Best: 151000 | Avg: 160924.20
Generasi 3 | Best: 151000 | Avg: 159414.50
Generasi 4 | Best: 149700 | Avg: 158122.60
Generasi 5 | Best: 148500 | Avg: 157000.60
Generasi 6 | Best: 148500 | Avg: 156193.90
Generasi 7 | Best: 147400 | Avg: 154593.00
Generasi 8 | Best: 146100 | Avg: 153055.40
Generasi 9 | Best: 144000 | Avg: 151325.40
Generasi 10 | Best: 141500 | Avg: 149209.10
Generasi 11 | Best: 141200 | Avg: 147202.80
Generasi 12 | Best: 139500 | Avg: 145047.00
Generasi 13 | Best: 137700 | Avg: 143626.20
Generasi 14 | Best: 137600 | Avg: 142412.40
Generasi 15 | Best: 136700 | Avg: 141318.90
Generasi 16 | Best: 134200 | Avg: 140136.20
Generasi 17 | Best: 134200 | Avg: 139333.70
Generasi 18 | Best: 133600 | Avg: 138437.00
Generasi 19 | Best: 130900 | Avg: 137469.90
Generasi 20 | Best: 130700 | Avg: 136338.10
Generasi 21 | Best: 129100 | Avg: 135586.80
Generasi 22 | Best: 128600 | Avg: 134403.4